# BNP Paribas × Société Générale — pairs research
Explore adjusted daily prices, the Engle–Granger test, rolling spread z-scores, and a simple paper backtest. A historical relationship is not evidence of future profitability.


In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
sys.path.insert(0, str(root))
import matplotlib.pyplot as plt
from src.data import download_adjusted_closes
from src.analytics import fit_hedge_ratio, engle_granger_test, rolling_zscore, kalman_hedge_ratio
from src.backtest import backtest_pair, performance_summary


In [ ]:
prices = download_adjusted_closes(start="2021-01-01")
prices.tail()


In [ ]:
fit = fit_hedge_ratio(prices)
test = engle_granger_test(prices)
print(f"OLS hedge ratio: {fit['beta']:.3f}")
print(f"Engle–Granger p-value: {test['p_value']:.4f}")
fit['spread'].plot(title="OLS residual spread")
plt.show()


In [ ]:
zscore = rolling_zscore(fit["spread"], window=60)
result = backtest_pair(prices, zscore, fit["beta"], entry_z=2.0, exit_z=0.5, transaction_cost_bps=5)
performance_summary(result)


In [ ]:
result["equity"].plot(title="Illustrative paper backtest equity curve")
plt.ylabel("Growth of 1 unit")
plt.show()


## Optional: dynamic hedge ratio
A Kalman filter allows the hedge ratio to vary over time, adding flexibility and potential overfitting risk. Compare it out of sample with the fixed OLS estimate.


In [ ]:
states = kalman_hedge_ratio(prices)
states["hedge_ratio"].plot(title="Time-varying hedge ratio")
plt.show()
